In [2]:
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from catboost import CatBoostClassifier

In [3]:
train_data = pd.read_csv("/content/train.csv")

In [4]:
train_data.head()

,area,perimeter,major_axis,minor_axis,eccentricity,eqdiasq,solidity,convex_area,extent,aspect_ratio,roundness,compactness,shapefactor_1,shapefactor_2,shapefactor_3,shapefactor_4,target
0,75516,1731.4840,411.7352,245.7620,0.8023,310.0806,0.9148,82546,0.7169,1.6753,0.3165,0.7531,0.0055,0.0033,0.5672,0.9502,1
1,98903,1374.4370,477.2451,269.7676,0.8249,354.8622,0.9585,103181,0.7679,1.7691,0.6579,0.7436,0.0048,0.0027,0.5529,0.9781,0
2,84746,1311.1570,482.7735,235.9040,0.8725,328.4843,0.9121,92914,0.7162,2.0465,0.6195,0.6804,0.0057,0.0028,0.4630,0.9474,1
3,98184,1463.1680,434.3769,292.6472,0.7390,353.5700,0.9543,102890,0.7316,1.4843,0.5763,0.8140,0.0044,0.0030,0.6625,0.9834,0
4,94170,1267.7271,440.1109,278.4162,0.7745,346.2672,0.9643,97656,0.6836,1.5808,0.7363,0.7868,0.0047,0.0030,0.6190,0.9785,0


In [5]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1288 entries, 0 to 1287
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   area           1288 non-null   int64  
 1   perimeter      1288 non-null   float64
 2   major_axis     1288 non-null   float64
 3   minor_axis     1288 non-null   float64
 4   eccentricity   1288 non-null   float64
 5   eqdiasq        1288 non-null   float64
 6   solidity       1288 non-null   float64
 7   convex_area    1288 non-null   int64  
 8   extent         1288 non-null   float64
 9   aspect_ratio   1288 non-null   float64
 10  roundness      1288 non-null   float64
 11  compactness    1288 non-null   float64
 12  shapefactor_1  1288 non-null   float64
 13  shapefactor_2  1288 non-null   float64
 14  shapefactor_3  1288 non-null   float64
 15  shapefactor_4  1288 non-null   float64
 16  target         1288 non-null   int64  
dtypes: float64(14), int64(3)
memory usage: 171.2 KB


In [6]:
test_data = pd.read_csv("/content/test.csv")

In [7]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 430 entries, 0 to 429
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   area           430 non-null    int64  
 1   perimeter      430 non-null    float64
 2   major_axis     430 non-null    float64
 3   minor_axis     430 non-null    float64
 4   eccentricity   430 non-null    float64
 5   eqdiasq        430 non-null    float64
 6   solidity       430 non-null    float64
 7   convex_area    430 non-null    int64  
 8   extent         430 non-null    float64
 9   aspect_ratio   430 non-null    float64
 10  roundness      430 non-null    float64
 11  compactness    430 non-null    float64
 12  shapefactor_1  430 non-null    float64
 13  shapefactor_2  430 non-null    float64
 14  shapefactor_3  430 non-null    float64
 15  shapefactor_4  430 non-null    float64
dtypes: float64(14), int64(2)
memory usage: 53.9 KB


# Logistic Regression Base Line

## Data Preprocessing

In [8]:
X, y = train_data.drop("target", axis=1), train_data["target"]

In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

X_test = test_data.to_numpy()
X_test_scaled = scaler.transform(X_test)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


# Training Logistic Regression

In [11]:
logreg_model = LogisticRegression(class_weight="balanced", random_state=42)
logreg_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', random_state=42)

In [12]:
from sklearn.metrics import classification_report, roc_auc_score

# Predictions
y_pred = logreg_model.predict(X_val_scaled)
y_prob = logreg_model.predict_proba(X_val_scaled)[:, 1]

# Metrics
print("Classification Report:")
print(classification_report(y_val, y_pred))
print(f"\nROC-AUC Score: {roc_auc_score(y_val, y_prob):.3f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.86      0.85       116
           1       0.88      0.86      0.87       142

    accuracy                           0.86       258
   macro avg       0.86      0.86      0.86       258
weighted avg       0.86      0.86      0.86       258


ROC-AUC Score: 0.928


## Training Logistic Regression with Cross-Validation

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [14]:
logreg_model_cv = LogisticRegression(class_weight="balanced", random_state=42)
cv_scores = cross_val_score(logreg_model_cv, X_train_scaled, y_train, cv=cv, scoring="accuracy")

In [15]:
print("CV Accuracy Scores:", cv_scores)
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

CV Accuracy Scores: [0.86893204 0.86407767 0.88349515 0.89805825 0.82524272]
Mean CV Accuracy: 0.8680 (±0.0244)


In [16]:
logreg_model_cv.fit(X_train_scaled, y_train)
print(f"Test Accuracy: {logreg_model_cv.score(X_val_scaled, y_val):.4f}")

Test Accuracy: 0.8605


In [17]:
y_pred = logreg_model_cv.predict(X_val_scaled)
print("Classification Report:")
print(classification_report(y_val, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.86      0.85       116
           1       0.88      0.86      0.87       142

    accuracy                           0.86       258
   macro avg       0.86      0.86      0.86       258
weighted avg       0.86      0.86      0.86       258



# Training CatBoost

In [18]:
from catboost import Pool

train_pool = Pool(X_train_scaled, y_train)
test_pool = Pool(X_val_scaled, y_val)

In [19]:
catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)


catboost_model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)

0:	learn: 0.8427184	test: 0.8062016	best: 0.8062016 (0)	total: 53.3ms	remaining: 53.3s
100:	learn: 0.9252427	test: 0.8643411	best: 0.8682171 (17)	total: 488ms	remaining: 4.34s
200:	learn: 0.9699029	test: 0.8643411	best: 0.8682171 (17)	total: 1.21s	remaining: 4.83s
300:	learn: 0.9834951	test: 0.8643411	best: 0.8682171 (17)	total: 2.33s	remaining: 5.41s
400:	learn: 0.9951456	test: 0.8565891	best: 0.8682171 (17)	total: 3.29s	remaining: 4.92s
500:	learn: 1.0000000	test: 0.8682171	best: 0.8682171 (17)	total: 3.83s	remaining: 3.82s
600:	learn: 1.0000000	test: 0.8565891	best: 0.8682171 (17)	total: 4.24s	remaining: 2.82s
700:	learn: 1.0000000	test: 0.8488372	best: 0.8682171 (17)	total: 4.68s	remaining: 2s
800:	learn: 1.0000000	test: 0.8488372	best: 0.8682171 (17)	total: 5.09s	remaining: 1.27s
900:	learn: 1.0000000	test: 0.8527132	best: 0.8682171 (17)	total: 5.53s	remaining: 608ms
999:	learn: 1.0000000	test: 0.8488372	best: 0.8682171 (17)	total: 5.93s	remaining: 0us

bestTest = 0.8682170543
bes

In [20]:
# Predictions
y_pred = catboost_model.predict(X_val_scaled)
y_prob = catboost_model.predict_proba(X_val_scaled)[:, 1]

# Metrics
print("Test Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:")
print(classification_report(y_val, y_pred))
print("\nROC-AUC:", roc_auc_score(y_val, y_prob))

Test Accuracy: 0.8682170542635659

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.84      0.85       116
           1       0.87      0.89      0.88       142

    accuracy                           0.87       258
   macro avg       0.87      0.87      0.87       258
weighted avg       0.87      0.87      0.87       258


ROC-AUC: 0.9322486644001943


## Training CatBoost with Optuna

In [21]:
use_class_weight = True

In [22]:
# 1. Define the objective function for Optuna
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
        'loss_function': 'Logloss',
        'eval_metric': 'Accuracy',
        'verbose': False,
        'random_state': 42
    }

    # Add class weight if needed
    if use_class_weight:
        params['scale_pos_weight'] = trial.suggest_float('scale_pos_weight', 0.5, 2)

    model = CatBoostClassifier(**params)
    model.fit(
        X_train,
        y_train,
        early_stopping_rounds=50,
        eval_set=(X_val, y_val),
        verbose=0
    )

    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

# 2. Run the optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, timeout=3600)

# 3. Get best parameters
print("Best trial:")
trial = study.best_trial
print(f"  Accuracy: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# 4. Train final model with best parameters
best_params = trial.params
best_params.update({
    'loss_function': 'Logloss',
    'eval_metric': 'Accuracy',
    'random_state': 42,
    'verbose': 100
})

final_model = CatBoostClassifier(
    **best_params,
    feature_border_type='GreedyLogSum',
    max_ctr_complexity=2,
)

final_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50,
    use_best_model=True
)

[I 2025-04-28 12:22:21,138] A new study created in memory with name: no-name-78ff0bdc-609f-4870-b923-d3a1c5b028e6
[I 2025-04-28 12:22:23,821] Trial 0 finished with value: 0.875968992248062 and parameters: {'iterations': 1143, 'learning_rate': 0.01187874789378622, 'depth': 9, 'l2_leaf_reg': 4.466532091319717, 'border_count': 175, 'random_strength': 4.309940990191201, 'bagging_temperature': 0.608071195596989, 'grow_policy': 'Depthwise', 'scale_pos_weight': 1.2185472103473203}. Best is trial 0 with value: 0.875968992248062.
[I 2025-04-28 12:22:26,628] Trial 1 finished with value: 0.8488372093023255 and parameters: {'iterations': 1033, 'learning_rate': 0.0021815773694915446, 'depth': 10, 'l2_leaf_reg': 2.6585930460091114, 'border_count': 219, 'random_strength': 7.8052087749751005, 'bagging_temperature': 0.5147038584347274, 'grow_policy': 'Lossguide', 'scale_pos_weight': 0.5600902526715493}. Best is trial 0 with value: 0.875968992248062.
[I 2025-04-28 12:22:27,032] Trial 2 finished with val

Best trial:
  Accuracy: 0.8915
  Params: 
    iterations: 1841
    learning_rate: 0.07827444903749116
    depth: 9
    l2_leaf_reg: 1.8784373516312929
    border_count: 67
    random_strength: 4.55315274533793
    bagging_temperature: 0.5828179339737369
    grow_policy: Depthwise
    scale_pos_weight: 1.148939412210376
0:	learn: 0.8674846	test: 0.8320884	best: 0.8320884 (0)	total: 17.9ms	remaining: 33s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8938262663
bestIteration = 6

Shrink model to first 7 iterations.


In [23]:
best_params

{'iterations': 1841,
 'learning_rate': 0.07827444903749116,
 'depth': 9,
 'l2_leaf_reg': 1.8784373516312929,
 'border_count': 67,
 'random_strength': 4.55315274533793,
 'bagging_temperature': 0.5828179339737369,
 'grow_policy': 'Depthwise',
 'scale_pos_weight': 1.148939412210376,
 'loss_function': 'Logloss',
 'eval_metric': 'Accuracy',
 'random_state': 42,
 'verbose': 100}

### Evaluate the best model

In [24]:
y_pred = final_model.predict(X_val)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.85      0.88       116
           1       0.89      0.92      0.90       142

    accuracy                           0.89       258
   macro avg       0.89      0.89      0.89       258
weighted avg       0.89      0.89      0.89       258



### Save the best results

In [25]:
y_pred_answers = final_model.predict(X_test)

In [26]:
answers = pd.DataFrame(y_pred)
answers.to_csv('answers.csv', index=False, header=False)